## 音声認識モデル

In [1]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

: 

In [1]:
import whisper
model = whisper.load_model("large-v3")

In [2]:

from whisper.decoding import BeamSearchDecoder
from typing import Tuple
import torch
import torch.nn.functional as F
from torch import Tensor
from whisper.tokenizer import get_tokenizer
import pandas as pd
from pathlib import Path

In [ ]:

# from whisper.decoding import BeamSearchDecoder
# from typing import Tuple
# import torch
# import torch.nn.functional as F
# from torch import Tensor
# from whisper.tokenizer import get_tokenizer
# import pandas as pd

# # トークナイザーの取得
# tokenizer = get_tokenizer(multilingual=True, language="ja", task="transcribe")

# candidates = []

# def beam_update(self, tokens: Tensor, logits: Tensor, sum_logprobs: Tensor) -> Tuple[Tensor, bool]:
#     if tokens.shape[0] % self.beam_size != 0:
#         raise ValueError(f"{tokens.shape}[0] % {self.beam_size} != 0")

#     n_audio = tokens.shape[0] // self.beam_size
#     if self.finished_sequences is None:
#         self.finished_sequences = [{} for _ in range(n_audio)]

#     logprobs = F.log_softmax(logits.float(), dim=-1)
#     next_tokens, source_indices, finished_sequences = [], [], []
#     last_context_detected = [False] * self.beam_size # Track '傷' detection for each beam

#     for i in range(n_audio):
#         scores, sources, finished = {}, {}, {}

#         for j in range(self.beam_size):
#             idx = i * self.beam_size + j
#             prefix = tokens[idx].tolist()
#             for logprob, token in zip(*logprobs[idx].topk(self.beam_size + 1)):
#                 new_logprob = (sum_logprobs[idx] + logprob).item()
#                 sequence = tuple(prefix + [token.item()])
#                 candidate_text = tokenizer.decode(list(sequence))
#                 candidates.append(sequence)

#                 # Context-based adjustments
#                 if "あ、" in candidate_text:
#                     last_context_detected[j] = True

#                 if last_context_detected[j] and "傷" in candidate_text.split():
#                     if "病" in candidate_text.split():
#                         new_logprob += 3000  # Boost significantly if '傷病' follows the context

#                 scores[sequence] = new_logprob
#                 sources[sequence] = idx

#         saved = 0
#         for sequence, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True):
#             next_tokens.append(sequence)
#             source_indices.append(sources[sequence])
#             saved += 1
#             if saved == self.beam_size:
#                 break

#         finished_sequences.append(finished)

#     tokens = torch.tensor(next_tokens, device=tokens.device)
#     self.inference.rearrange_kv_cache(source_indices)
#     completed = all(len(finished) >= self.max_candidates for finished in finished_sequences)
#     return tokens, completed

# BeamSearchDecoder.update = beam_update



## gennzyouno betuo

In [70]:
import whisper
from whisper.decoding import BeamSearchDecoder
from typing import Tuple
import torch
import torch.nn.functional as F
from torch import Tensor
from whisper.tokenizer import get_tokenizer
import pandas as pd

# トークナイザーの取得
tokenizer = get_tokenizer(multilingual=True, language="ja", task="transcribe")

candidates = []

# 傷病者に優先順位をつけたカスタムビーム更新機能
def beam_update(self, tokens: Tensor, logits: Tensor, sum_logprobs: Tensor) -> Tuple[Tensor, bool]:
    if tokens.shape[0] % self.beam_size != 0:
        raise ValueError(f"{tokens.shape}[0] % {self.beam_size} != 0")

    n_audio = tokens.shape[0] // self.beam_size
    if self.finished_sequences is None:  
        self.finished_sequences = [{} for _ in range(n_audio)]

    logprobs = F.log_softmax(logits.float(), dim=-1)
    next_tokens, source_indices, finished_sequences = [], [], []
    for i in range(n_audio):
        scores, sources, finished = {}, {}, {}

        # 可能性のある候補の累積対数確率を計算
        for j in range(self.beam_size):
            idx = i * self.beam_size + j
            prefix = tokens[idx].tolist()
            for logprob, token in zip(*logprobs[idx].topk(self.beam_size + 1)):
                new_logprob = (sum_logprobs[idx] + logprob).item()
                sequence = tuple(prefix + [token.item()])
                candidates.append(sequence)
                scores[sequence] = new_logprob
                sources[sequence] = idx


        # 候補をランク付けし、各オーディオのビーム・サイズ・シーケンスの上位をキープ
        saved = 0
        for sequence in sorted(scores, key=scores.get, reverse=True):
            candidate_text = tokenizer.decode(list(sequence))
            
            # 傷病者の確率
            if "傷病者" in candidate_text:
                scores[sequence]  += 10
             

                
            if sequence[-1] == self.eot:
                finished[sequence] = scores[sequence]
            else:
                sum_logprobs[len(next_tokens)] = scores[sequence]
                next_tokens.append(sequence)
                source_indices.append(sources[sequence])

                saved += 1
                if saved == self.beam_size:
                    break

        finished_sequences.append(finished)

    tokens = torch.tensor(next_tokens, device=tokens.device)
    self.inference.rearrange_kv_cache(source_indices)

    # add newly finished sequences to self.finished_sequences
    assert len(self.finished_sequences) == len(finished_sequences)
    for previously_finished, newly_finished in zip(
        self.finished_sequences, finished_sequences
    ):
        for seq in sorted(newly_finished, key=newly_finished.get, reverse=True):
            if len(previously_finished) >= self.max_candidates:
                break  # the candidate list is full
            previously_finished[seq] = newly_finished[seq]

    # mark as completed if all audio has enough number of samples
    completed = all(
        len(sequences) >= self.max_candidates
        for sequences in self.finished_sequences
    )
    return tokens, completed

# Replace the original update method with the custom one
BeamSearchDecoder.update = beam_update


In [8]:
import whisper
from whisper.decoding import BeamSearchDecoder
from typing import Tuple
import torch
import torch.nn.functional as F
from torch import Tensor
from whisper.tokenizer import get_tokenizer
import pandas as pd

# トークナイザーの取得
tokenizer = get_tokenizer(multilingual=True, language="ja", task="transcribe")

candidates = []

# 傷病者に優先順位をつけたカスタムビーム更新機能
def beam_update(self, tokens: Tensor, logits: Tensor, sum_logprobs: Tensor) -> Tuple[Tensor, bool]:
    if tokens.shape[0] % self.beam_size != 0:
        raise ValueError(f"{tokens.shape}[0] % {self.beam_size} != 0")

    n_audio = tokens.shape[0] // self.beam_size
    if self.finished_sequences is None:  
        self.finished_sequences = [{} for _ in range(n_audio)]

    logprobs = F.log_softmax(logits.float(), dim=-1)
    next_tokens, source_indices, finished_sequences = [], [], []
    for i in range(n_audio):
        scores, sources, finished = {}, {}, {}
        max_score = float('-inf')  # 初期値は非常に小さな数値

        # 可能性のある候補の累積対数確率を計算
        for j in range(self.beam_size):
            idx = i * self.beam_size + j
            prefix = tokens[idx].tolist()
            for logprob, token in zip(*logprobs[idx].topk(self.beam_size + 1)):
                new_logprob = (sum_logprobs[idx] + logprob).item()
                sequence = tuple(prefix + [token.item()])
                candidates.append(sequence)
                scores[sequence] = new_logprob
                sources[sequence] = idx
                candidate_text = tokenizer.decode(list(sequence))

                if scores[sequence] > max_score:
                    max_score = scores[sequence]

        for sequence, score in scores.items():
            candidate_text = tokenizer.decode(list(sequence))
            if "傷病者" in candidate_text:
                scores[sequence] = max_score + 1  # 現行の最大スコアを超えるように調整

        # 候補をランク付けし、各オーディオのビーム・サイズ・シーケンスの上位をキープ
        saved = 0
        for sequence in sorted(scores, key=scores.get, reverse=True):
            if sequence[-1] == self.eot:
                finished[sequence] = scores[sequence]
            else:
                sum_logprobs[len(next_tokens)] = scores[sequence]
                next_tokens.append(sequence)
                source_indices.append(sources[sequence])

                saved += 1
                if saved == self.beam_size:
                    break

        finished_sequences.append(finished)

    tokens = torch.tensor(next_tokens, device=tokens.device)
    self.inference.rearrange_kv_cache(source_indices)

    # add newly finished sequences to self.finished_sequences
    assert len(self.finished_sequences) == len(finished_sequences)
    for previously_finished, newly_finished in zip(
        self.finished_sequences, finished_sequences
    ):
        for seq in sorted(newly_finished, key=newly_finished.get, reverse=True):
            if len(previously_finished) >= self.max_candidates:
                break  # the candidate list is full
            previously_finished[seq] = newly_finished[seq]

    # mark as completed if all audio has enough number of samples
    completed = all(
        len(sequences) >= self.max_candidates
        for sequences in self.finished_sequences
    )
    return tokens, completed

# Replace the original update method with the custom one
BeamSearchDecoder.update = beam_update


In [9]:
# Transcribe the audio file with the custom beam search decoder
result_beam = model.transcribe("/app/whisper/2回目_川村先生.wav", beam_size=3, initial_prompt="傷 病 者")


In [10]:
result_beam

{'text': '2回目の手技を行いますあ 傷病者発見 周囲は安全です 感染防御に配慮します近づいていって 意識の確認をします大丈夫ですか 大丈夫ですか 大丈夫ですか 大丈夫ですか意識はない 誰か誰か誰か来てください1人目のあなた あなた119番通報してください2人目のあなた あなた近くにあるAEDを持ってきてください必ずここに戻ってきてください胸とお腹を見て 呼吸の確認 同時に脈の確認10秒以内 5 6 7 呼吸脈ありません胸骨圧迫を開始しますいきます2 3 4 5 6 7 8 9 102 2 3 4 5 6 7 8 9 103 2 3 4 5 6 7 8 9 101 2 3 4 5 6 7 8 9 102 2 3 4 5 6 7 891032345678910あ、えーとAEDを持ってきてくれたあなたあなたAEDの使い方分かりますか?分からないじゃあこの胸骨圧迫を変わってください123で変わりましょう123胸骨圧迫続けてくださいAEDが来たらまず最初に電源を入れますパッドを胸に装着してくださいパッドを取り出して装着していきます体表面を接続してくださいパッドを装着してくださいコネクタを接続してください心電図の解析をしますので胸骨圧迫を知っていてあなた離れてくださいショックします安全確認をします私よしあなたよし周りよしあなたよしショックしますショック胸骨圧迫を直ちに開始します3456789102234567891032345678910救急隊のあなたこの人が5分前に目の前で倒れるのを目撃しました胸骨圧迫をしてAEDで1回ショックをしていますこの人の身元はわかりませんがこの人の荷物はそこにあります以上ですはいはい',
 'segments': [{'id': 0,
   'seek': 0,
   'start': 0.0,
   'end': 5.44,
   'text': '2回目の手技を行います',
   'tokens': [50365,
    17,
    8350,
    11386,
    2972,
    11389,
    32502,
    5998,
    8082,
    11267,
    50637],
   'temperature': 0.0,
   'avg_logprob': 0.633910966956097

In [74]:

# セグメントごとのトークンとデコード結果の抽出
token_data = []
for segment in result_beam['segments']:
    tokens = segment['tokens']
    decoded_results = [tokenizer.decode([token]) for token in tokens]
    token_data.extend(zip(tokens, decoded_results))

# データフレームの作成
df = pd.DataFrame(token_data, columns=['Token', 'Decoded Result'])
df.to_csv("test.csv")

In [ ]:
[(cand[4:], tokenizer.decode(cand[4:])) for cand in candidates]

In [68]:
result_beam

{'text': '2回目の手技を行いますあ 症病者発見 周囲は安全です 感染防御に配慮します近づいていって 意識の確認をします大丈夫ですか 大丈夫ですか 大丈夫ですか 大丈夫ですか意識はない 誰か誰か誰か来てください1人目のあなた あなた119番通報してください2人目のあなた あなた近くにあるAEDを持ってきてください必ずここに戻ってきてください胸とお腹を見て 呼吸の確認 同時に脈の確認10秒以内 5 6 7 呼吸脈ありません胸骨圧迫を開始します1 2 3 4 5 6 7 8 9 102 2 3 4 5 6 7 8 9 103 2 3 4 5 6 7 8 9 101 2 3 4 5 6 7 8 9 102 2 3 4 5 6 7 891032345678910あ、えーとAEDを持ってきてくれたあなたあなたAEDの使い方分かりますか?分からないじゃあこの胸骨圧迫を変わってください123で変わりましょう123胸骨圧迫続けてくださいAEDが来たらまず最初に電源を入れますパッドを胸に装着してくださいパッドを取り出して装着していきます体表面を接続してくださいパッドを装着してくださいコネクタを接続してください心電図を解析中です体に触れないでくださいえーと心電図の解析をしますので胸骨圧迫を知っていてあなた離れてくださいショックが必要です充電中です体から離れてくださいショックを実行します安全確認をします私よしあなたよし周りよしあなたよしショックしますショックショックが完了しました胸骨圧迫を直ちに開始します3456789102234567891032345678910あ、救急隊のあなたえーとこの人が5分前に目の前で倒れるのを目撃しましたえーとこの人が5分前に目の前で倒れるのを目撃しましたえーと胸骨圧迫をしてAEDで1回ショックをしていますこの人の身元はわかりませんがこの人の荷物はそこにあります以上ですはい',
 'segments': [{'id': 0,
   'seek': 0,
   'start': 0.0,
   'end': 5.42,
   'text': '2回目の手技を行います',
   'tokens': [50365,
    17,
    8350,
    11386,
    2972,
    11389,
    32502,
    5998,


In [77]:
correct_words = ["安全", '傷病者発見', '感染防御', '大丈夫ですか', '誰か来てください', '119番', 'AED', '呼吸の確認', "脈の確認", "1 2 3", 'ショック', '荷物']

# result_beamからテキストを取得
text = result_beam['text']

# 各単語がテキストに含まれているかを確認
word_presence = {word: word in text for word in correct_words}

# 結果を表示
df = pd.DataFrame(list(word_presence.items()), columns=['Word', 'Presence'])
df

,Word,Presence
0,安全,True
1,傷病者発見,True
2,感染防御,True
3,大丈夫ですか,True
4,誰か来てください,True
5,119番,True
6,AED,True
7,呼吸の確認,True
8,脈の確認,True
9,1 2 3,True


In [73]:

# Save all candidates to a file
output_dir = Path("outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "2transcription_candidates.txt", "w", encoding="utf-8") as f:
    for i, candidate in enumerate(candidates):
        candidate_text = tokenizer.decode(list(candidate))
        print(f"候補 {i+1}: {candidate_text}")
        f.write(f"候補 {i+1}: {candidate_text}\n")

# Display and save the top-k candidates
k = 5  # Number of top candidates to display
topk_candidates = candidates[:k]
topk_decoded_words = [tokenizer.decode(list(candidate)) for candidate in topk_candidates]
df = pd.DataFrame(topk_decoded_words, columns=["テキスト"])

# Save the DataFrame to a CSV file
df.to_csv("test.csv", index=False, encoding="utf-8")

print(df)

候補 1: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 2: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 3: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 4: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 5: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 6: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 7: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 8: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 9: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 10: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 11: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 12: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>
候補 13: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>2
候補 14: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>二
候補 15: <|nospeech|> 傷 病 者<|startoftranscript|><|ja|><|startoflm|>小
候補 16: <|nospeec

In [64]:
correct_words = ["周囲の安全よし",'傷病者発見','感染防御', '大丈夫ですか', '誰か来てください','119番', 'AED','呼吸の確認',"脈の確認","1 2 3",'蘇生','荷物']

In [ ]:

# すべての候補を表示し、ファイルに保存
output_dir = Path("outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "candidates.txt", "w", encoding="utf-8") as f:
    for i, candidate in enumerate(candidates):
        candidate_text = tokenizer.decode(list(candidate))
        print(f"候補 {i+1}: {candidate_text}")
        f.write(f"候補 {i+1}: {candidate_text}\n")

# 上位k候補を表示し、ファイルに保存
k = 5 # 表示したい上位候補の数
topk_candidates = candidates[:k]

topk_decoded_words = [tokenizer.decode(list(candidate)) for candidate in topk_candidates]
df = pd.DataFrame(topk_decoded_words, columns=["テキスト"])

# DataFrameをファイルに保存
df.to_csv(output_dir / "top_candidates.csv", index=False, encoding="utf-8")

print(df)

# 2

In [ ]:
from whisper.decoding import BeamSearchDecoder
from typing import Tuple, List
import torch
import torch.nn.functional as F
from torch import Tensor
from whisper.tokenizer import get_tokenizer
import pandas as pd
import logging
import whisper
from whisper.decoding import GreedyDecoder
from typing import Tuple
import torch
import torch.nn.functional as F
from torch import Tensor
from torch.distributions import Categorical

seq_logits = []
def update(self, tokens: Tensor, logits: Tensor, sum_logprobs: Tensor) -> Tuple[Tensor, bool]:
    temperature = self.temperature
    seq_logits.append(logits)
    if temperature == 0:
        next_tokens = logits.argmax(dim=-1)
    else:
        next_tokens = Categorical(logits=logits / temperature).sample()

    logprobs = F.log_softmax(logits.float(), dim=-1)
    current_logprobs = logprobs[torch.arange(logprobs.shape[0]), next_tokens]
    sum_logprobs += current_logprobs * (tokens[:, -1] != self.eot)

    next_tokens[tokens[:, -1] == self.eot] = self.eot
    tokens = torch.cat([tokens, next_tokens[:, None]], dim=-1)

    completed = (tokens[:, -1] == self.eot).all()
    return tokens, completed

GreedyDecoder.update = update

In [ ]:

# Transcribe the audio file with the custom beam search decoder
beam = model.transcribe("/app/whisper/2回目_川村先生.wav", initial_prompt="傷 病 者")

In [ ]:
beam